# CopyLane 판정 인코더 파인튜닝 v8.1 — 3차 골든 · 학습 단위 비교(유형 ↔ 호) (박수진, 2026-10-05)

> 골든 `529c970556d7` · 8,383행(학습 5,894 · 평가 2,489) · 사전 506종 · 근거: **D-296 개정 2** · D-318 · D-301 · D-311 · 전달 `docs/ohb/전달_2026-10-05_소성민·박수진_인코더_재학습.md`
> 🔴 **v7.x 수치와 비교하지 않는다** — 골든 · 평가셋이 다르다. 기준선은 이 노트북의 `all · 낱말 포함` 실행이다.

## v7.5 → v8 에서 바꾼 것

| # | 바꾼 것 | 근거 |
|:-:|---|---|
| 1 | **학습 행을 조건 칸으로 고른다** — 아래 표 | D-296 개정 2 |
| 2 | 골든 sha · 행 수(8,383 · 5,894 · 2,489)를 대조한다. 다르면 멈춘다 | D-176 |
| 3 | `기능성화장품_오인`(화장품법 13①2 · 학습 11행)을 **보조 로짓**에 더했다 — 확정 6종은 그대로 · 저장 모델은 6로짓 | D-232 · 전달 §5 「측정 불가」 |
| 4 | **낱말 행(327) 스위치** `USE_WORD_ROWS` — 넣은 판 · 뺀 판을 dev 로 비교한다 | 팀장 지시 10-03 |
| 5 | **적법 문장 오탐을 내역과 함께** 낸다 — 주장 있음(조건 L) · 주장 없음(수정문구 판 D · id `gf:`). 해설서 위반문구 D 는 참고 | D-301 |
| 6 | 저장 정보(`label_scheme.json`)에 **조건별 학습 행 수 · 뺀 행 수 · 사전 sha** 를 남긴다 | 전달 §4-3 · 전달 10-02 §3 |
| 7 | 저장 이름을 하나로 — `copylane-encoder-kcbert-v8-<REASON_MODE>[-낱말제외].zip` (Colab · Drive 같음) | |
| 8 | 실험 로그 파일을 나눴다 — `copylane_v8_experiment_log.json` | 평가셋이 달라 v7 로그와 섞지 않는다 |
| 9 | **dev 는 문장 행만** — 낱말 행 · 유형 없는 위반은 dev 채점에서 뺀다. 그래서 스위치를 바꿔도 dev 가 **같은 행**이다 | 판끼리 바로 비교하려면 같은 행으로 재야 한다 |
| 10 | **dev 음성 = 유형이 하나도 없는 행** — 보조 유형만 있는 위반 행은 음성에서 뺀다(v7.x 는 음성으로 세었다) | 위반 문구에 울린 것을 오판정으로 세지 않는다 |

모델 구조 · 손실 · dev 분리 · threshold 방식 · test 봉인(D-175)은 v7.5 와 같다. 이전 판 설명은 v7.5 노트북 머리말에 있다.

## 🆕 v8.1 — 호로 학습해 유형으로 접는 판 (`LABEL_UNIT = "ho"`)

전달 10-05 §5 의 물음 「학습은 호로 하되 건넬 때 유형으로 접는 판과 처음부터 유형으로 학습한 판을 둘 다 재서 고를 수 있는가」에 답하려는 실험이다. **dev 로만 비교한다.**

| | 유형 판 (`"type"` · 1 · 2회차) | 호 판 (`"ho"` · 3회차) |
|---|---|---|
| 학습 로짓 | 유형 9칸 (확정 6 + 보조 3) | 학습에 양성이 있는 **호 전부** (식품 5호 다목은 따로 — 후기) |
| 정답 | 골든의 `labels` | 골든의 `근거` 의 호 |
| 유형 없는 위반(화장품법 13①4 등) | 학습에서 뺀다 | **그 호의 양성으로 쓴다** (근거도 없는 5행만 뺀다) |
| 채점 | 유형 그대로 | 호 확률을 **유형으로 접어서**(같은 유형의 호 중 최댓값) 유형 판과 같은 dev 행 · 같은 지표로 |
| 저장 | 6로짓 모델 (앱 로더 형식) | 🔴 저장하지 않는다 — 접기를 추론 때 해야 한다. 고르게 되면 형식을 팀장과 정한다 |

- 섹션 2 가 「호 → 유형 대조」를 찍는다 — 노트북의 대응표가 골든의 유형 칸과 한 행이라도 다르면 호 판은 멈춘다(`collect/statute.py` 와 같아야 한다).
- dev 행은 유형 판과 **같다**. 유형 없는 위반은 dev 채점에 없으므로, dev 묶음에 든 그 행들(어느 판도 학습하지 않는다)의 **탐지율을 참고**로 따로 찍는다.

### 호 판을 고르는 규칙 (초안 — 🔴 돌리기 전에 확정한다 · 비교 기준은 같은 설정의 유형 판)

1. 탈락 — 아래 G1 ~ G3 · B2 를 하나라도 어기면 유형 판을 둔다.
2. 호 판을 고르는 조건 — 통과하고, **dev macro PR-AUC 가 0.015 이상 오르거나 dev 음성 오판정이 3줄 이상 줄 때**.
3. 그 밖에는 유형 판을 둔다 — 앱 로더와 바로 맞고 단순하다. 유형 없는 위반 탐지율은 참고다(행 수가 적다).

## 학습 행 고르기 (섹션 4 첫머리가 집행한다)

| 학습 행 | 처리 | 기대 행 수 |
|---|---|---|
| 조건 **M** | **뺀다** — 양성도 음성도 아니다 | 654 (그중 유형이 있는 것 541) |
| 조건 D · L | 음성 (유형 전부 0) | 612 |
| 조건 C · A · B · 조건 없음 — 유형 있음 | 그 유형이 양성 | 4,450 |
| 조건 C · A · B — **유형 없음** | **뺀다** (`DROP_UNTYPED_POS = True`) | 178 (근거도 없는 것 5) |

- 🔴 **유형 없는 위반 178행** — 화장품법 13①4 처럼 호만으로 유형을 못 정하는 위반이다. 유형 6종 모델에서는 양성으로 넣을 자리가 없고, 음성으로 넣으면 위반 문구를 「위반 아님」으로 가르친다. 그래서 뺀다. ⬜ 출력 단위(호 ↔ 유형) 판정과 함께 팀장 확인 사항이다.
- 조건 없는 학습 행(사전 낱말 · 주입 1,111)은 전부 유형이 있다.

## 실행 순서

| 회차 | 설정 (섹션 3) | 볼 것 |
|:-:|---|---|
| 1 | `USE_WORD_ROWS = True` · `FINAL_TEST = False` | 섹션 4 의 학습 행 표가 위 기대 수와 같은지 · 섹션 6 dev 지표 |
| 2 | `USE_WORD_ROWS = False` · `FINAL_TEST = False` | 섹션 9 비교표 · 사전 고정 기준 자동 판정 |
| **3** | **`LABEL_UNIT = "ho"` · `USE_WORD_ROWS = False`** · `FINAL_TEST = False` ← 🟢 지금 설정 | 섹션 2 호 → 유형 대조 0 · 섹션 9 자동 판정(기준 = 2회차) |
| 4 | 고른 판 하나만 `FINAL_TEST = True` | 🔴 test 는 이 한 번만 (D-175) |

- 업로드(섹션 1): `golden.jsonl` · `banned_terms.jsonl` 둘뿐이다.
- 🔴 **추가 파일(4호 주입 · D형 음성)은 쓰지 않는다** — `EXTRA_TRAIN_FILES = []`. 3차 골든만으로 먼저 잰다(10-05). 나중에 붙일 때는 `proto_negd4.jsonl` 을 새 분할로 다시 만든다(`gen_negd4.py` · D-174).
- 판을 바꿀 때는 섹션 3 스위치만 고치고 **섹션 3부터 끝까지** 다시 실행한다.

## 사전 고정 기준 (초안 — 🔴 돌리기 전에 확정한다 · 비교 기준은 같이 돌린 `all · 낱말 포함`)

| # | 조건 | 어디서 |
|---|---|---|
| G1 | dev 위반 재현율 하락 2%p 이내 | 섹션 9 자동 |
| G2 | dev 양성 n≥30 유형의 재현율 하락 5%p 이내 | 섹션 9 자동 |
| G3 | dev 음성 오판정 증가 1줄 이내 | 섹션 9 자동 |
| G4 | 4호 홀드아웃 탐지 하락 5%p 이내 — 추가 파일을 쓸 때만(이번에는 해당 없음) | 섹션 9 자동 |
| G5 | 판정 로직 dev 양성 놓침(통과) 0 | 로컬 `run_judge_dist.py` |
| B2 | dev macro PR-AUC(시드 평균) 하락 0.02 이내 | 섹션 9 자동 |
| test | 최종 후보 하나만 1회 — 기준은 dev 결과를 보기 전에 따로 적는다 | 별도 |

## ⚠️ 알고 볼 것

- **dev 는 train 과 같은 원천이다** — dev 점수가 test 성능을 과대추정한다(v7.5: dev 재현율 98.7% → test 81.4%). 후보는 dev 로 고르되 test 기준을 미리 정한다.
- dev 채점에 이유 구역 행은 넣지 않는다(v7 그대로). 이유 구역은 판독으로 호가 문구 단위가 됐지만(D-318) 판독이 전부 모델이라 dev 구성은 바꾸지 않았다.
- D-301 — 평가 D 의 **해설서 위반문구 행은 적법 문장이 아니다**(기관이 삭제를 지시한 문구). 그 플래그율은 오탐률이 아니라 참고 수치다.
- 3호(표시광고법 부당 비교 · 학습 14) · 화장품법 2호(학습 11)는 이번 재학습에서 「측정 불가」다.
- `redistributable: false` 행이 섞여 있다. 노트북 출력과 산출물을 외부 공개 저장소에 올리지 않는다.


## 1. 환경 설정 + 파일 업로드

Colab에서 `sympy` 충돌로 `from transformers import ...`가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.
런타임에 같은 이름 파일이 이미 있으면 새로 올린 파일은 `golden (1).jsonl`로 저장되고 코드는 예전 파일을 읽는다. 새 판을 올릴 때는 런타임을 새로 시작한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random
from collections import Counter, defaultdict
import numpy as np

from google.colab import files as colab_files
print("① golden.jsonl 선택")
uploaded = dict(colab_files.upload())
print("② banned_terms.jsonl 선택")
uploaded.update(colab_files.upload())
print("업로드된 파일:", list(uploaded.keys()))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다. 새 판을 쓸 때는 data/derived_manifest.jsonl 의 sha 로 이 값을 바꾼다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # v8 — 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가 (전달 10-05 §1)
EXPECTED_BANNED_SHA = "455f8d7ea5c227ffab268a33cef9bc36750b0144f073a2ed33e1f7298dc5b892"   # 사전 506종
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "새 판이면 EXPECTED_GOLDEN_SHA 를 derived_manifest 값으로 바꾼다")
BANNED_SHA = sha256("banned_terms.jsonl")
print("banned_terms.jsonl sha256:", BANNED_SHA[:16], "…" + ("" if BANNED_SHA == EXPECTED_BANNED_SHA else "  🟡 원장(506종)과 다르다 — 기준선 B 가 달라진다. data-sync 한 파일인지 확인"))


## 2. 데이터 로드 + eval 조건 버킷 — D-174 · D-232 · D-286 · D-301

`split` 필드를 그대로 따른다(D-174). 학습 행 고르기는 섹션 4 첫머리에서 한다(스위치를 읽어야 해서).

| 버킷 | 정의 | 기대 응답 | 재는 것 |
|---|---|---|---|
| `scored` | 확정 6종 라벨 있음 · 보조 라벨 없음 · 조건 ≠ M · D | 위반 (그 유형) | 라벨별 P/R/F1 · ROC/PR-AUC · G1 |
| `pending` | 라벨 없음 · 조건 C/A/B (화장품법 4호 등 유형 없는 위반) | 위반 (유형 미정) | 위반 탐지 재현율 |
| `aux` | 비방 · 부당비교 · 기능성화장품 오인 라벨 있음 | (확정 6종 밖) | 참고만 |
| `neg` | 라벨 없음 · 조건 L 또는 조건 없음 | 위반 없음 | **적법 문장 오탐 — 주장 있음** |
| `D` | 조건 D | 판정 대상 아님 | id `gf:` = **적법 문장 오탐 — 주장 없음** · 그 밖(해설서 위반문구)은 참고 |
| `M` | 조건 M | `low_conf` 보류 | 채점 안 함 (참고) |


In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

golden = load_jsonl("golden.jsonl")
print("golden.jsonl:", len(golden), "행")

# D-232 — 인코더가 예측하는 확정 클래스 (scripts/collect.py VIOLATION_TYPES 순서)
CONFIRMED = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]
AUX = ["부당_비교광고", "비방광고", "기능성화장품_오인"]   # 확정 6종 밖 — 학습용 보조 로짓으로만 (v8: 화장품법 13①2 추가)
LAW_NAMES = {"002011": "표시광고법", "013094": "식품표시광고법", "002015": "화장품법"}

def norm_text(s):
    return " ".join(s.split())

def group_key(r):
    # 같은 원문에서 나온 행을 한 묶음으로 — dev 분할 단위
    rid = r["id"]
    if rid.startswith("inj:"):             # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def to_ho(c):
    return c.split("|")[0]

# 🆕 v8.1 — 호 → 유형 대응표. collect/statute.py 의 _HO · _MOK 와 같아야 한다(아래에서 골든의 유형 칸으로 대조한다)
_F, _C, _K = "013094:제8조제1항제", "002011:제3조제1항제", "002015:제13조제1항제"
HO_TYPE = {
    _F + "1호": "질병_예방치료_표방", _F + "2호": "의약품_오인", _F + "3호": "건강기능식품_오인", _F + "4호": "거짓_과장",
    _F + "5호": "소비자_기만", _F + "5호|다목": "후기_체험기_기만", _F + "6호": "비방광고", _F + "7호": "부당_비교광고",
    _C + "1호": "거짓_과장", _C + "2호": "소비자_기만", _C + "3호": "부당_비교광고", _C + "4호": "비방광고",
    _K + "1호": "의약품_오인", _K + "2호": "기능성화장품_오인",
}   # 여기 없는 호(화장품법 13①4 · 식품 8 ~ 10호)는 유형이 없다

def ho_key(c):
    # 학습 단위 = 호. 목이 유형을 바꾸는 자리(식품 5호 다목 = 후기)만 목까지 든다
    return c if c in HO_TYPE else c.split("|")[0]

def normalize(r):
    basis = r.get("근거") or []
    return {
        "id": r["id"], "group": group_key(r), "text": r["text"],
        "labels": r.get("labels") or [], "split": r["split"],
        "provenance": r.get("provenance"), "origin": r.get("origin"),
        "zone": r.get("구역"), "condition": r.get("조건"), "unit": r.get("unit"),
        "basis": basis, "hos": sorted({to_ho(c) for c in basis}), "hokeys": sorted({ho_key(c) for c in basis}),
        "laws": sorted({c.split(":")[0] for c in basis}),
    }

all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(CONFIRMED) - set(AUX)
assert not unknown, f"예상 밖 라벨: {unknown}"
# 🆕 v8.1 — 호를 유형으로 접은 것이 골든의 유형 칸과 같은가 (대응표 검증 · 호 판은 어긋나면 멈춘다)
HO_MISMATCH = [r["id"] for r in all_rows if sorted({HO_TYPE[k] for k in r["hokeys"] if k in HO_TYPE}) != sorted(r["labels"])]
print(f"호 → 유형 대조: 어긋난 행 {len(HO_MISMATCH)}" + (f" · 예 {HO_MISMATCH[:3]}" if HO_MISMATCH else " (골든의 유형 칸과 같다)"))

train_all = [r for r in all_rows if r["split"] == "train"]
eval_rows = [r for r in all_rows if r["split"] != "train"]
_got = (len(all_rows), len(train_all), len(eval_rows))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 전달 문서와 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")

def train_role(r):
    # 🆕 v8 (D-296 개정 2) — 학습 행을 조건 칸으로 읽는다. 집행은 섹션 4 첫머리(스위치를 읽는다)
    #    M = 뺀다(양성도 음성도 아니다) · D · L = 음성 · C · A · B · 조건 없음 = 유형이 양성
    #    C · A · B 인데 유형이 없으면(화장품법 13①4 · 근거 결손) 유형 6종으로는 양성 자리가 없다 → "pos_untyped"
    c = r["condition"]
    if c == "M":
        return "drop_M"
    if c in ("D", "L"):
        return "neg"
    if r["labels"]:
        return "pos"
    return "pos_untyped" if c in ("C", "A", "B") else "neg"

def lawful_kind(r):
    # 🆕 v8 (D-301 · preprocess/golden.py lawful_kind 와 같은 규칙) — 적법 문장은 둘뿐이다
    if not r["labels"] and (r["condition"] or "L") == "L":
        return "주장"          # 원천이 위반 아님을 선언 (조건 L)
    if r["condition"] == "D" and r["id"].startswith("gf:"):
        return "주장없음"      # 원천이 승인한 형태 · 주장 없음 (해설서 수정문구 판)
    return None

_tc = Counter((train_role(r), r["condition"] or "없음") for r in train_all)
print("학습 행 (조건 × 역할):", {f"{c}:{ro}": n for (ro, c), n in sorted(_tc.items(), key=lambda kv: (kv[0][1], kv[0][0]))})
print(f"  M {sum(n for (ro, c), n in _tc.items() if c == 'M')} (유형 있는 것 {sum(1 for r in train_all if r['condition'] == 'M' and r['labels'])}) · "
      f"D·L {sum(n for (ro, c), n in _tc.items() if c in ('D', 'L'))} · 유형 없는 C·A·B {sum(n for (ro, c), n in _tc.items() if ro == 'pos_untyped')} · "
      f"낱말 행 {sum(1 for r in train_all if r['unit'] == '낱말')}   ← 기대: 654(541) · 612 · 178 · 327")

def eval_bucket(r):
    # 팀장 채점 기준(10-01) — C·A·B = 양성(scored + pending) · L = 음성(neg) · D = 별도 지표(D-301 내역) · M = 채점 안 함
    #    조건이 M · D 면 라벨보다 먼저 가른다(채점 밖). L 인데 라벨이 있으면 데이터가 어긋난 것이라 멈춘다.
    if r["condition"] == "M":
        return "M"
    if r["condition"] == "D":
        return "D"
    if r["condition"] == "L":
        assert not r["labels"], f"🔴 조건 L(적법)인데 라벨이 있다 — {r['id']} · {r['labels']}"
        return "neg"
    if any(l in AUX for l in r["labels"]):
        return "aux"
    if r["labels"]:
        return "scored"
    if r["condition"] in ("C", "B", "A"):
        return "pending"
    return "neg"

for r in eval_rows:
    r["bucket"] = eval_bucket(r)
BUCKETS = {b: [r for r in eval_rows if r["bucket"] == b] for b in ["scored", "aux", "neg", "D", "M", "pending"]}
print(f"train {len(train_all)} / eval {len(eval_rows)}")
print(f"  팀장 채점: 양성(C·A·B) {len(BUCKETS['scored']) + len(BUCKETS['pending'])} · 음성(L) {len(BUCKETS['neg'])} · "
      f"D(별도) {len(BUCKETS['D'])} · M(채점 안 함) {len(BUCKETS['M'])} · aux(참고) {len(BUCKETS['aux'])}")
for b, rows in BUCKETS.items():
    print(f"  {b:8s}: {len(rows):5d}  출처={dict(Counter(r['provenance'] for r in rows))}")
_lk = Counter(lawful_kind(r) for r in eval_rows)
print(f"  적법 문장(D-301): 주장 있음 {_lk['주장']} · 주장 없음 {_lk['주장없음']} · 해설서 위반문구 D(적법 아님) {len(BUCKETS['D']) - _lk['주장없음']}")

## 3. 실험 스위치 — 이 셀만 바꾸고 「섹션 3부터 끝까지」 다시 실행

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `LABEL_UNIT` | `"ho"` | 🆕 v8.1 학습 단위 — `"type"`(유형 · 1 · 2회차) / `"ho"`(호로 학습해 유형으로 접는다 · 3회차) |
| `USE_WORD_ROWS` | `False` | 🆕 사례집 **낱말 행(327)** 을 학습에 넣을지. `False` 면 뺀 판 — 둘을 dev 로 비교한다 |
| `DROP_UNTYPED_POS` | `True` | 🆕 유형 없는 위반 행(조건 C·A·B · 178)을 학습에서 뺀다. `False` 면 음성으로 넣는다(v7 방식 · 권하지 않는다) |
| `REASON_MODE` | `"all"` | 이유 구역 처리 — `all`(D-234) / `weighted` / `exclude` / `label_mask` / `mask_nonclaim`. v8 기본은 `all` |
| `USE_AUX_LOGITS` | `True` | 비방 · 부당비교 · 기능성화장품 오인을 보조 로짓으로 학습할지 |
| `USE_INJECTED` | `True` | 주입 행(D-25)을 학습에 넣을지 |
| `DEDUP` | `True` | train 안 같은 문구 병합 |
| `SEEDS` | `[42, 43, 44]` | 시드마다 학습 한 번 |
| `FINAL_TEST` | `False` | 🔴 test 채점은 최종 후보 1회만 (D-175) |
| `SAVE_THIS_RUN` | `True` | 모델 zip 저장 — 로컬 판정 분포(`run_judge_dist.py`)에 쓴다 |
| `EXTRA_TRAIN_FILES` | `[]` | 🔄 추가 학습 파일 — 이번 3차 측정은 없이 돌린다 |

조건 M 은 스위치가 없다 — 언제나 뺀다(D-296 개정 2).


In [ ]:
BACKBONE = "kcbert"                  # "kcbert"(D-70) / "kcelectra"(D-84)
LABEL_UNIT = "ho"                    # 🆕 v8.1 — 학습 단위. "type"(유형 · 1 · 2회차) / "ho"(호로 학습 → 유형으로 접어 채점 · 3회차)
USE_WORD_ROWS = False                # 사례집 낱말 행(327). 2회차 결과로 뺀 판이 후보다(dev 10-05) — 3회차도 뺀 채로 비교한다
DROP_UNTYPED_POS = True              # 🆕 v8 — 유형 없는 위반 행(조건 C·A·B · 178)을 학습에서 뺀다. False = 음성으로 넣는다(v7 방식)
REASON_MODE = "all"                  # "all"(D-234 · v8 기본) / "weighted" / "exclude" / "label_mask" / "mask_nonclaim"
REASON_WEIGHT = 0.4                  # weighted일 때만
REASON_MASK_LABELS = ["거짓_과장", "소비자_기만"]   # label_mask · mask_nonclaim일 때
# v7.3 — mask_nonclaim: 이유 구역 중 짧고(NONCLAIM_MAXLEN 미만) 주장 어휘가 없는 행만 마스킹
NONCLAIM_MAXLEN = 25
NONCLAIM_CUE = (r"[!?~]|효과|효능|개선|회복|예방|치료|완치|최고|최초|유일|1위|No\.?1|넘버원|보장|무료|할인|특가|최저|저렴|혁명|명품|"
                r"프리미엄|완벽|확실|안전|인증|특허|수상|추천|인기|만족|보증|증명|입증|검증|공인|대박|파격|절약|혜택|수익|확정|평생|"
                r"100%|\d+\s*%|배|만원|억|원\b|싸|좋|최|초|강력|탁월|우수|뛰어|놀라|기적|비밀|특별|독점|단독|전문|정품|천연|무첨가|"
                r"친환경|역세권|대단지|분양|임대")
USE_AUX_LOGITS = True
USE_INJECTED = True
DEDUP = True

SEEDS = [42, 43, 44]
DEV_FRAC, DEV_SEED = 0.15, 42        # dev 분할은 시드와 무관하게 고정
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
USE_FOCAL_LOSS, FOCAL_GAMMA = True, 1.5
POS_WEIGHT_CAP, POS_WEIGHT_SQRT = 5, True

SAVE_THIS_RUN = True                # 모델 zip 저장 (로컬 판정 분포에 쓴다)
FINAL_TEST = False                   # v7.4 — 🔴 test 채점은 최종 후보 1회만 (D-175). 실험 비교는 dev 로
EXTRA_HOLDOUT_FRAC = 0.2             # v7.4 — 추가 파일마다 같은 20% 를 떼어 홀드아웃 지표로 잰다(어느 실행에서도 학습 안 함)
THRESHOLD_MODE = "best_seed"         # "best_seed"(저장 모델 자신의 dev threshold · 기본) / "robust"(시드 평균 F1 최대 · 참고용)

# v7.1 — train에만 붙일 추가 jsonl (실험 전용 · 채택은 팀장 판정). None이면 v7과 같다
EXTRA_TRAIN_FILES = []              # 🔄 v8 — 3차 골든 첫 측정은 추가 파일 없이 (10-05). 붙이려면 ["proto_inj4c.jsonl", "proto_negd4.jsonl"] · negd4 는 새 분할로 다시 만든다
EVAL_ONLY_FILES = []                # 학습하지 않고 홀드아웃 지표만 잴 파일 — 기준 실행에서 쓴다

BACKBONE_MODEL_MAP = {"kcbert": "beomi/kcbert-base", "kcelectra": "beomi/KcELECTRA-base"}
MODEL_NAME = BACKBONE_MODEL_MAP[BACKBONE]
assert LABEL_UNIT in ("type", "ho"), LABEL_UNIT
if LABEL_UNIT == "ho" and HO_MISMATCH:
    raise SystemExit(f"🔴 호 → 유형 대응표가 골든과 다르다 — {len(HO_MISMATCH)}행 · 예 {HO_MISMATCH[:5]}. HO_TYPE 을 collect/statute.py 와 맞춘다")
UNTYPED = "유형없음"                 # 호 판에서만 — 유형이 없는 호(화장품법 13①4 등)가 접히는 칸
# 채점 공간(유형). 호 판은 학습 로짓(호)을 이 칸들로 접는다 — 섹션 4 의 fold()
LABEL_LIST = (CONFIRMED + AUX + [UNTYPED]) if LABEL_UNIT == "ho" else CONFIRMED + (AUX if USE_AUX_LOGITS else [])
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_CONF = len(CONFIRMED)              # 앞 6개가 확정 클래스

reason_tag = {"weighted": f"weighted{REASON_WEIGHT}",
              "label_mask": "mask(" + ",".join(REASON_MASK_LABELS) + ")",
              "mask_nonclaim": f"mask_nonclaim(<{NONCLAIM_MAXLEN})"}.get(REASON_MODE, REASON_MODE)
loss_tag = (f"focal(g={FOCAL_GAMMA},cap={POS_WEIGHT_CAP}{',sqrt' if POS_WEIGHT_SQRT else ''})"
            if USE_FOCAL_LOSS else f"bce(cap={POS_WEIGHT_CAP})")
def _flags(word=True, untyped_drop=True, unit="type"):
    # 실험 키의 뒷부분 — 기본 스위치(낱말 포함 · 유형 없는 위반 제외)면 "|v8" 로 끝난다
    f = ("" if USE_AUX_LOGITS else "|noaux") + ("" if USE_INJECTED else "|noinj") + ("" if DEDUP else "|nodedup")
    for _f in EXTRA_TRAIN_FILES:
        f += f"|+{os.path.splitext(os.path.basename(_f))[0]}"
    return f + "|v8" + ("" if word else "|낱말제외") + ("" if untyped_drop else "|유형없음=음성") + ("" if unit == "type" else "|호학습")
assert not set(EXTRA_TRAIN_FILES) & set(EVAL_ONLY_FILES), "같은 파일을 학습 · 평가 전용에 함께 둘 수 없다"
EXTRA_SHA = {}
for _f in list(EXTRA_TRAIN_FILES) + list(EVAL_ONLY_FILES):
    if not os.path.exists(_f):
        print(f"③ {_f} 선택")
        colab_files.upload()
    EXTRA_SHA[_f] = sha256(_f)
    print(f"{_f} sha256: {EXTRA_SHA[_f][:16]}…" + ("  (평가 전용)" if _f in EVAL_ONLY_FILES else ""))
flags = _flags(USE_WORD_ROWS, DROP_UNTYPED_POS, LABEL_UNIT)
_th_tag = "|th=robust" if THRESHOLD_MODE == "robust" else ""
exp_key = f"{BACKBONE}|이유={reason_tag}|{loss_tag}{flags}{_th_tag}"
# 비교 기준 — 호 판은 **같은 설정의 유형 판** · 그 밖은 all · 낱말 포함
BASE_KEY = (f"{BACKBONE}|이유={reason_tag}|{loss_tag}{_flags(USE_WORD_ROWS, DROP_UNTYPED_POS)}{_th_tag}" if LABEL_UNIT == "ho"
            else f"{BACKBONE}|이유=all|{loss_tag}{_flags()}{_th_tag}")
RUN_NAME = (f"copylane-encoder-{BACKBONE}-v8-{REASON_MODE}" + ("" if USE_WORD_ROWS else "-낱말제외")
            + ("" if DROP_UNTYPED_POS else "-유형없음음성") + ("" if LABEL_UNIT == "type" else "-호학습"))   # 저장 zip 이름
print("exp_key =", exp_key)
print("저장 이름 =", RUN_NAME + ".zip", "· 비교 기준 =", "이 실행" if exp_key == BASE_KEY else BASE_KEY)
print("LABEL_LIST =", LABEL_LIST)

LOG_PATH = "/content/drive/MyDrive/copylane_v8_experiment_log.json"   # v8 — 평가셋이 달라 v7 로그와 나눈다
if "EXPERIMENT_LOG" not in globals():
    EXPERIMENT_LOG = {}
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, encoding="utf-8") as f:
                EXPERIMENT_LOG = json.load(f)
            print(f"Drive에서 이전 실험 {len(EXPERIMENT_LOG)}건 복원:", list(EXPERIMENT_LOG))
    except Exception as e:
        print("이전 실험 로그 복원 안 함:", e)

## 4. 학습 행 고르기 + dev 분리 + 학습셋 구성

0. 🆕 **학습 행 고르기 (D-296 개정 2)** — 조건 M 을 빼고 · D · L 을 음성으로 · 유형 없는 위반과 (스위치에 따라) 낱말 행을 뺀다. 표가 머리말의 기대 수와 같은지 본다.
1. **dev 분리** — train의 묶음(문서 · 원본 문구) 단위로, 확정 6종마다 「이유가 아닌 양성」의 약 15%가 dev에 가도록 층화해서 뽑는다. 음성 묶음도 15%를 넣는다.
2. dev 채점은 그 묶음의 **이유가 아닌 행**으로만 한다. dev 묶음의 이유 행은 train에서도 뺀다(같은 문서가 양쪽에 서지 않게).
3. `DEDUP` — train 안 같은 문구(같은 구역 종류끼리)를 한 행으로 합치고 라벨은 합집합으로 둔다. dev 와 같은 문구는 train에서 뺀다.
4. `REASON_MODE`에 따라 행 가중치(`sample_weight`)와 라벨 마스크(`loss_mask`)를 정한다.


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def devable(r):
    # dev 채점 행 — 이유 구역은 뺀다(v7 그대로). 🆕 v8: 낱말 행 · 유형 없는 위반도 뺀다
    #   → dev 가 스위치(USE_WORD_ROWS · DROP_UNTYPED_POS)와 무관하게 **같은 행**이라 판끼리 바로 비교된다. 문장 단위 모델은 문장으로 잰다
    return not is_reason(r) and r["unit"] != "낱말" and r.get("role") != "pos_untyped"

# ── 0) 🆕 v8 학습 행 고르기 (D-296 개정 2) — M 제외 · D · L 음성 · 유형 없는 위반 · 낱말 행은 스위치
#    train_use = 조건 M 만 뺀 전부(dev 분리의 모집단 — 스위치와 무관). 스위치는 아래 2) train 풀에서 건다
train_use, ROLE_COUNTS = [], Counter()
for r in train_all:
    role = train_role(r)
    shown = role
    if role == "pos_untyped":
        if LABEL_UNIT == "ho" and r["hokeys"]:
            shown = "pos_호만"       # 🆕 v8.1 — 호 판은 유형 없는 호도 그 호의 양성으로 쓴다
        else:
            shown = "drop_유형없음" if DROP_UNTYPED_POS else "neg"
    if r["unit"] == "낱말" and not USE_WORD_ROWS and not shown.startswith("drop"):
        shown = "drop_낱말"
    ROLE_COUNTS[f"{shown}|{r['condition'] or '없음'}"] += 1
    if role == "drop_M":
        continue
    kind = "drop" if shown.startswith("drop") else ("pos" if shown.startswith("pos") else "neg")   # 이 판에서의 쓰임
    train_use.append(dict(r, role=role, kind=kind, labels=list(r["labels"]) if role == "pos" else []))
_by = lambda p: sum(n for k, n in ROLE_COUNTS.items() if k.startswith(p))
print(f"학습 행 고르기: 전체 {len(train_all)} → 쓰는 행 {_by('pos') + _by('neg|')}  (양성 {_by('pos')} · 음성 {_by('neg|')})"
      + (f" · 그중 유형 없는 호를 양성으로 쓰는 행 {_by('pos_호만')}" if LABEL_UNIT == "ho" else ""))
print(f"  뺀 행 — 조건 M {_by('drop_M')} · 유형 없는 위반 {_by('drop_유형없음')} · 낱말 {_by('drop_낱말')}")
print("  내역:", dict(sorted(ROLE_COUNTS.items())))
assert not any(r["condition"] == "M" for r in train_use), "조건 M 이 학습에 남았다 (D-296 개정 2)"
assert not any(r["labels"] for r in train_use if r["condition"] in ("D", "L")), "조건 D · L 행에 유형이 남았다"

# ── 1) 묶음 단위 층화 dev 분리
groups = defaultdict(list)
for r in train_use:
    groups[r["group"]].append(r)
rng = random.Random(DEV_SEED)
gkeys = sorted(groups)
rng.shuffle(gkeys)

pos_total = Counter(l for r in train_use if devable(r) for l in r["labels"] if l in CONFIRMED)
target = {l: max(1, math.ceil(DEV_FRAC * pos_total[l])) for l in CONFIRMED}
neg_groups = [g for g in gkeys if any(devable(r) and not r["labels"] for r in groups[g])]
neg_target = math.ceil(DEV_FRAC * len(neg_groups))
got, neg_got, dev_groups = Counter(), 0, set()
for g in gkeys:
    labs = Counter(l for r in groups[g] if devable(r) for l in r["labels"] if l in CONFIRMED)
    need = any(got[l] < target[l] for l in labs)
    is_neg = g in neg_groups and not labs
    if need or (is_neg and neg_got < neg_target):
        dev_groups.add(g)
        got.update(labs)
        neg_got += is_neg

dev_rows = [r for g in dev_groups for r in groups[g] if devable(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
print(f"dev: 묶음 {len(dev_groups)}개 · 채점 행 {len(dev_rows)} (음성 {sum(1 for r in dev_rows if not r['labels'])})")
print("dev 양성:", {l: sum(l in r['labels'] for r in dev_rows) for l in CONFIRMED})
print("dev 음성 조건:", dict(Counter(r["condition"] or "없음" for r in dev_rows if not r["labels"])))
# 🆕 v8.1 — dev 묶음에 든 유형 없는 위반. 어느 판도 학습하지 않는다(묶음째 빠진다) → 탐지율을 참고로 잰다
DEV_UNTYPED = [r for g in dev_groups for r in groups[g] if r["role"] == "pos_untyped" and not is_reason(r) and r["unit"] != "낱말"]
print(f"dev 묶음의 유형 없는 위반: {len(DEV_UNTYPED)}행 (참고 지표 · 학습 안 함)")

# ── 2) train 풀
# 스위치(유형 없는 위반 · 낱말 행 · 학습 단위)는 0) 에서 정한 kind 에 들어 있다 — "drop" 만 뺀다
pool = [r for r in train_use if r["group"] not in dev_groups and r["kind"] != "drop"]
if not USE_INJECTED:
    pool = [r for r in pool if r["origin"] != "injected"]
before = len(pool)
# v7.4 — test 와 같은 문구는 train 에서 빼지 않는다(D-174 · 누수는 물질화가 평가 쪽에서 뺀다). dev 와 같은 문구만 뺀다
pool = [r for r in pool if norm_text(r["text"]) not in dev_texts]
print(f"train 풀: {before} → dev와 같은 문구 제거 후 {len(pool)}")

# ── 3) 같은 문구 병합 (이유/비이유는 따로 — 마스크 규칙이 다르므로)
if DEDUP:
    merged = {}
    for r in pool:
        k = (norm_text(r["text"]), is_reason(r))
        if k in merged:
            m = merged[k]
            m["labels"] = sorted(set(m["labels"]) | set(r["labels"]))
            if r["kind"] == "pos":      # 🆕 v8.1 — 호 라벨도 합집합 · 양성이 하나라도 있으면 양성
                m["hokeys"] = sorted(set(m["hokeys"]) | set(r["hokeys"]))
                m["kind"] = "pos"
        else:
            merged[k] = dict(r, labels=list(r["labels"]), hokeys=list(r["hokeys"]) if r["kind"] == "pos" else [])
    pool = list(merged.values())
    print(f"같은 문구 병합 후 {len(pool)}행")

# ── 3.5) v7.1 추가 train 파일 — dev · test · 기존 train과 같은 문구는 버린다
extra_rows, extra_holdout = [], {}   # extra_holdout: 파일 → 홀드아웃 행 (학습 안 함)
_have = {norm_text(r["text"]) for r in pool}
for _f in sorted(set(EXTRA_TRAIN_FILES) | set(EVAL_ONLY_FILES)):
    _all = load_jsonl(_f)
    # family 별 · id 순 · 시드 고정으로 같은 EXTRA_HOLDOUT_FRAC 를 뗀다 — 학습 여부와 무관하게 같은 행이다
    _fam = defaultdict(list)
    for r in _all:
        _fam[r.get("family", "-")].append(r)
    _hold, _hrng = [], random.Random(DEV_SEED)
    for fam, rs in sorted(_fam.items()):
        rs = sorted(rs, key=lambda r: r["id"])
        _hrng.shuffle(rs)
        _hold += rs[:int(round(EXTRA_HOLDOUT_FRAC * len(rs)))]
    extra_holdout[_f] = [normalize(r) for r in _hold]
    if _f not in EXTRA_TRAIN_FILES:
        print(f"평가 전용 {_f}: 홀드아웃 {len(_hold)}행 (학습 안 함)")
        continue
    _hid, _n0 = {r["id"] for r in _hold}, len(extra_rows)
    for r in _all:
        if r["id"] in _hid:
            continue
        x = normalize(r)
        t = norm_text(x["text"])
        if t in dev_texts or t in _have:
            continue
        assert set(x["labels"]) <= set(LABEL_LIST), x["labels"]
        x["kind"] = "pos" if x["labels"] else "neg"
        if LABEL_UNIT == "ho" and x["labels"] and not x["hokeys"]:
            raise SystemExit(f"🔴 {_f}: 근거(호)가 없는 양성 행이다 — 호 판에는 붙일 수 없다 ({x['id']})")
        _have.add(t)
        extra_rows.append(x)
    print(f"추가 train {_f}: {len(extra_rows) - _n0}행 · 홀드아웃 {len(_hold)}행 · 라벨 {dict(Counter(l for r in extra_rows[_n0:] for l in r['labels']))}")
pool = pool + extra_rows

# ── 4) REASON_MODE
if REASON_MODE == "exclude":
    pool = [r for r in pool if not is_reason(r)]
train_rows = pool
train_weights = [REASON_WEIGHT if (REASON_MODE == "weighted" and is_reason(r)) else 1.0 for r in train_rows]
# 🆕 v8.1 — 학습 로짓의 단위. 유형 판: 유형(확정 6 + 보조) · 호 판: 학습에 양성이 있는 호 전부
if LABEL_UNIT == "ho":
    TRAIN_LABELS = sorted({k for r in train_use if r["role"] in ("pos", "pos_untyped") for k in r["hokeys"]})
    TRAIN_TYPE = [HO_TYPE.get(k, UNTYPED) for k in TRAIN_LABELS]     # 로짓마다 접히는 유형
else:
    TRAIN_LABELS, TRAIN_TYPE = list(LABEL_LIST), list(LABEL_LIST)
TRAIN2ID = {k: i for i, k in enumerate(TRAIN_LABELS)}
FOLD_IDX = [[i for i, t in enumerate(TRAIN_TYPE) if t == l] for l in LABEL_LIST]

def fold(p):
    # 학습 로짓 공간(행, 호) → 채점 공간(행, 유형 = LABEL_LIST). 한 유형에 호가 여럿이면 가장 큰 값. 유형 판에서는 그대로다
    if LABEL_UNIT != "ho":
        return p
    out = np.zeros((p.shape[0], len(LABEL_LIST)), dtype=np.float32)
    for j, idx in enumerate(FOLD_IDX):
        if idx:
            out[:, j] = p[:, idx].max(axis=1)
    return out

mask_idx = [i for i, t in enumerate(TRAIN_TYPE) if t in REASON_MASK_LABELS]
import re as _re
_cue = _re.compile(NONCLAIM_CUE)
def is_nonclaim_reason(r):
    t = r["text"].strip()
    return is_reason(r) and len(t) < NONCLAIM_MAXLEN and not _cue.search(t)
if REASON_MODE == "mask_nonclaim":
    _nc = [r for r in train_rows if is_nonclaim_reason(r)]
    print(f"mask_nonclaim: 이유 {sum(map(is_reason, train_rows))}행 중 {len(_nc)}행의 {REASON_MASK_LABELS} 손실을 끈다 · 예: {[r['text'] for r in _nc[:6]]}")

def to_multihot(labels):
    v = np.zeros(len(LABEL_LIST), dtype=np.float32)
    for l in labels:
        if l in LABEL2ID:
            v[LABEL2ID[l]] = 1.0
    return v

def to_target(r):
    # 학습 정답 — 유형 판은 유형 칸 · 호 판은 근거의 호(양성 행만)
    if LABEL_UNIT != "ho":
        return to_multihot(r["labels"])
    v = np.zeros(len(TRAIN_LABELS), dtype=np.float32)
    if r.get("kind") == "pos":
        for k in r["hokeys"]:
            v[TRAIN2ID[k]] = 1.0
    return v

def row_mask(r):
    m = np.ones(len(TRAIN_LABELS), dtype=np.float32)
    if (REASON_MODE == "label_mask" and is_reason(r)) or (REASON_MODE == "mask_nonclaim" and is_nonclaim_reason(r)):
        m[mask_idx] = 0.0
    return m

w = np.array(train_weights)
Y = np.stack([to_target(r) for r in train_rows])
M = np.stack([row_mask(r) for r in train_rows])
WM = w[:, None] * M
TRAIN_BY_COND = dict(sorted(Counter(r["condition"] or "없음" for r in train_rows).items()))   # 저장 정보 (전달 10-05 §4-3)
print(f"\nREASON_MODE={REASON_MODE}: train {len(train_rows)}행 (이유 {sum(map(is_reason, train_rows))}행) · 조건별 {TRAIN_BY_COND}")
print(f"{'라벨':20s} {'학습 행':>8s} {'양성':>7s} {'양성 비율':>8s}")
for i, l in enumerate(TRAIN_LABELS):
    d, p = WM[:, i].sum(), (WM[:, i] * Y[:, i]).sum()
    tag = f"  → {TRAIN_TYPE[i]}" if LABEL_UNIT == "ho" else ("" if i < N_CONF else "  (보조)")
    print(f"  {l:28s} {d:8.0f} {p:7.0f} {p / max(d, 1e-6) * 100:7.1f}%{tag}")

# ── 토크나이즈
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(rows, weights=None, masks=None):
    ds = Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "labels": [to_target(r) for r in rows],
        "sample_weight": list(weights) if weights is not None else [1.0] * len(rows),
        "loss_mask": list(masks) if masks is not None else [np.ones(len(TRAIN_LABELS), dtype=np.float32)] * len(rows),
    })
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "sample_weight", "loss_mask"])
    return ds

train_ds = to_ds(train_rows, train_weights, M)
dev_ds = to_ds(dev_rows)
print("train_ds:", len(train_ds), "/ dev_ds:", len(dev_ds), "· 학습 로짓", len(TRAIN_LABELS), "칸 →", "유형으로 접어 채점" if LABEL_UNIT == "ho" else "유형 그대로")
if LABEL_UNIT == "ho":   # 접은 정답이 유형 정답과 같아야 한다 — 다르면 채점이 어긋난다
    _a = fold(np.stack([to_target(r) for r in dev_rows]))[:, :N_CONF]
    _b = np.stack([to_multihot(r["labels"]) for r in dev_rows])[:, :N_CONF]
    assert (_a == _b).all(), f"🔴 dev 에서 접은 호 정답이 유형 정답과 다르다 — {int((_a != _b).any(1).sum())}행"

## 5. 손실함수 · 지표

- Focal Loss(α = pos_weight). 원소(행×라벨)마다 `sample_weight × loss_mask`로 가중 평균을 낸다. 꺼진 원소는 분자와 분모 모두에서 빠진다.
- pos_weight는 마스크를 반영한 가중 개수로 계산한다. `POS_WEIGHT_SQRT`면 √(음성/양성)을 1~CAP으로 자른다.
- dev 지표는 **확정 6종만** 본다. 선택 기준은 **dev macro PR-AUC**(dev 양성이 있는 유형 평균)다.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding, set_seed
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score, f1_score

pos = (WM * Y).sum(axis=0)
neg = WM.sum(axis=0) - pos
ratio = neg / np.clip(pos, 1e-6, None)
pos_weight = np.clip(np.sqrt(ratio) if POS_WEIGHT_SQRT else ratio, 1, POS_WEIGHT_CAP)
print("pos_weight:", {l: round(float(pos_weight[i]), 2) for i, l in enumerate(TRAIN_LABELS)})
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

def elementwise_loss(logits, labels):
    pw = pos_weight_tensor.to(logits.device)
    logits = logits.float()
    if USE_FOCAL_LOSS:
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        return (pw * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce
    return torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw, reduction="none")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sw = inputs.pop("sample_weight", None)
        lm = inputs.pop("loss_mask", None)
        outputs = model(**inputs)
        el = elementwise_loss(outputs.logits, labels.float())
        wt = torch.ones_like(el)
        if sw is not None:
            wt = wt * sw.to(el.dtype)[:, None]
        if lm is not None:
            wt = wt * lm.to(el.dtype)
        loss = (el * wt).sum() / wt.sum().clamp_min(1e-6)
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def conf_metrics(y, p):
    # 확정 6종만 — y, p 는 (행, 전체 로짓)
    out, prs, rocs = {}, [], []
    for i in range(N_CONF):
        pr, roc = safe(average_precision_score, y[:, i], p[:, i]), safe(roc_auc_score, y[:, i], p[:, i])
        out[f"prauc_{CONFIRMED[i]}"], out[f"rocauc_{CONFIRMED[i]}"] = pr, roc
        if pr is not None:
            prs.append(pr); rocs.append(roc)
    out["macro_prauc"] = float(np.mean(prs)) if prs else float("nan")
    out["macro_rocauc"] = float(np.mean(rocs)) if rocs else float("nan")
    _, _, f, _ = precision_recall_fscore_support(y[:, :N_CONF], (p[:, :N_CONF] > 0.5).astype(int), average="micro", zero_division=0)
    out["micro_f1_at05"] = f
    return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # 🆕 v8.1 — 호 판은 정답 · 확률을 유형으로 접은 뒤 잰다(유형 판에서는 fold 가 그대로 돌려준다) → 조기 종료 기준이 두 판에서 같다
    return conf_metrics(fold(labels).astype(int), fold(1 / (1 + np.exp(-logits))))

def pick_thresholds(y, p):
    # dev에서 유형별 F1 최대 · 하한 0.1 · dev 양성 10건 미만이면 0.5
    th = {}
    for i in range(N_CONF):
        if y[:, i].sum() < 10:
            th[CONFIRMED[i]] = 0.5
            continue
        best_f, best_t = -1, 0.5
        for t in np.arange(0.1, 0.9001, 0.025):
            f = f1_score(y[:, i], p[:, i] >= t, zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(round(t, 3))
        th[CONFIRMED[i]] = best_t
    return th

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=128,
                        padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return fold(np.concatenate(out)) if out else np.zeros((0, len(LABEL_LIST)))   # 🆕 v8.1 — 언제나 유형 공간으로 돌려준다

## 6. 학습 — 시드마다 한 번 (조기 종료)

시드마다 dev macro PR-AUC가 가장 높은 epoch의 가중치를 `./runs/<실험>/seed<시드>`에 저장하고, 그 dev에서 threshold를 고른다.

In [ ]:
slug = "".join(ch if ch.isalnum() else "_" for ch in exp_key)[:80]
RUN_DIR = f"./runs/{slug}"
collator = DataCollatorWithPadding(tokenizer)
dev_y = np.stack([to_multihot(r["labels"]) for r in dev_rows]).astype(int)
seed_results = {}

for seed in SEEDS:
    print(f"\n==================== seed {seed} ====================")
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(TRAIN_LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(TRAIN_LABELS)), label2id=TRAIN2ID)
    args = TrainingArguments(
        output_dir=f"{RUN_DIR}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=0.01, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                              data_collator=collator, compute_metrics=compute_metrics,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()

    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    dm = conf_metrics(dev_y, dev_p)
    th = pick_thresholds(dev_y, dev_p)
    save_dir = f"{RUN_DIR}/seed{seed}"
    trainer.save_model(save_dir)
    shutil.rmtree(f"{RUN_DIR}/ckpt{seed}", ignore_errors=True)
    seed_results[seed] = {"dev": dm, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
                          "thresholds": th, "dir": save_dir, "dev_p": dev_p, "dev_u_p": predict_probs(trainer.model, DEV_UNTYPED),
                          "curve": [(int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)) for h in hist]}
    print(f"seed {seed}: dev macro PR-AUC {dm['macro_prauc']:.3f} (best epoch {seed_results[seed]['best_epoch']}) · 곡선 {seed_results[seed]['curve']}")
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

devs = [v["dev"]["macro_prauc"] for v in seed_results.values()]
BEST_SEED = max(seed_results, key=lambda s: seed_results[s]["dev"]["macro_prauc"])
print(f"\n[{exp_key}] dev macro PR-AUC {np.mean(devs):.3f} ± {np.std(devs):.3f}  (시드 {SEEDS}) · 가장 좋은 시드 {BEST_SEED}")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>8s}" for s in SEEDS))
for l in CONFIRMED:
    vals = [seed_results[s]["dev"].get(f"prauc_{l}") for s in SEEDS]
    print(f"{l:20s} " + " ".join(f"{v:8.3f}" if v is not None else "     N/A" for v in vals))

# ── v7.2 robust threshold: 시드마다의 dev 예측에서 F1을 구해 평균이 가장 높은 threshold
def pick_thresholds_robust(y, probs_list):
    th = {}
    for i in range(N_CONF):
        if y[:, i].sum() < 10:
            th[CONFIRMED[i]] = 0.5
            continue
        grid = np.arange(0.1, 0.9001, 0.025)
        mean_f = [np.mean([f1_score(y[:, i], p[:, i] >= t, zero_division=0) for p in probs_list]) for t in grid]
        th[CONFIRMED[i]] = float(round(grid[int(np.argmax(mean_f))], 3))
    return th

TH_ROBUST = pick_thresholds_robust(dev_y, [seed_results[s]["dev_p"] for s in SEEDS])
print(f"\nthreshold (THRESHOLD_MODE={THRESHOLD_MODE})")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>7s}" for s in SEEDS) + f" {'robust':>8s}")
for l in CONFIRMED:
    print(f"{l:20s} " + " ".join(f"{seed_results[s]['thresholds'][l]:7.3f}" for s in SEEDS) + f" {TH_ROBUST[l]:8.3f}")

# ── dev 조건 지표 (가장 좋은 시드 · 그 시드의 dev threshold) — 실험 비교는 이것으로 (D-175)
def wilson(k, n, z=1.96):
    """Wilson 95% 신뢰구간 (D-40)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

_th_dev = np.array([seed_results[BEST_SEED]["thresholds"][l] for l in CONFIRMED])
_dp = seed_results[BEST_SEED]["dev_p"][:, :N_CONF]
_dy = dev_y[:, :N_CONF]
_pos = _dy.any(1)
_flag = (_dp >= _th_dev).any(1)
# 🔄 v8 — dev 음성은 **유형이 하나도 없는 행**만이다. 보조 유형(비방 · 부당비교 · 기능성화장품)만 있는 행은 위반이라 음성에서 뺀다
#    (v7.x 는 이 행들을 음성으로 세었다 — 위반 문구에 울린 것이 「음성 오판정」으로 잡혔다)
_neg = np.array([not r["labels"] for r in dev_rows], dtype=bool)
dev_cond = {"dev_violation_recall": float(_flag[_pos].mean()) if _pos.any() else float("nan"),
            "dev_neg_fpr": float(_flag[_neg].mean()) if _neg.any() else float("nan"),
            "n_pos": int(_pos.sum()), "n_neg": int(_neg.sum()), "n_aux_only": int((~_pos & ~_neg).sum()),
            "k_pos_hit": int(_flag[_pos].sum()), "k_neg_flag": int(_flag[_neg].sum())}  # 줄 수(사전 고정 기준 G1 · G3)
_dc = np.array([r["condition"] or "없음" for r in dev_rows])
for _c in sorted(set(_dc[_neg])):   # 🆕 v8 — dev 음성 오판정을 조건별로 (D · L · 없음)
    _m = _neg & (_dc == _c)
    dev_cond[f"n_neg:{_c}"] = int(_m.sum()); dev_cond[f"k_neg_flag:{_c}"] = int(_flag[_m].sum())
_up = seed_results[BEST_SEED]["dev_u_p"]   # 🆕 v8.1 — dev 묶음의 유형 없는 위반(학습 안 함)을 얼마나 잡나 · 참고
if len(_up):
    _uflag = (_up[:, :N_CONF] >= _th_dev).any(1)
    if LABEL_UNIT == "ho":      # 유형 없는 호의 로짓 — 문턱을 고를 dev 양성이 없어 0.5 로 둔다
        _uflag = _uflag | (_up[:, LABEL2ID[UNTYPED]] >= 0.5)
    dev_cond["n_untyped"] = int(len(_up)); dev_cond["k_untyped_hit"] = int(_uflag.sum())
if extra_holdout:
    _best_tmp = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
    if torch.cuda.is_available():
        _best_tmp = _best_tmp.cuda()
    for _f, _rows in extra_holdout.items():
        if not _rows:
            continue
        _hp = predict_probs(_best_tmp, _rows)[:, :N_CONF] >= _th_dev
        _stem = os.path.splitext(os.path.basename(_f))[0]
        if any(r["labels"] for r in _rows):   # 라벨 있는 파일 — 정답 유형 탐지율 (높을수록 좋다)
            _y = np.stack([to_multihot(r["labels"]) for r in _rows])[:, :N_CONF].astype(bool)
            dev_cond[f"hold:{_stem}:탐지율"] = float((_hp & _y).any(1).mean())
            dev_cond[f"k_hold:{_stem}"] = int((_hp & _y).any(1).sum())
        else:                                  # 음성 파일 — 6종 중 하나라도 울린 비율 (낮을수록 좋다)
            dev_cond[f"hold:{_stem}:오탐률"] = float(_hp.any(1).mean())
        dev_cond[f"n_hold:{_stem}"] = len(_rows)
    del _best_tmp
print(f"\n[dev 조건 지표 · 시드 {BEST_SEED}]")
for k, v in dev_cond.items():
    print(f"  {k:22s} {v*100:6.1f}%" if isinstance(v, float) else f"  {k:22s} {v}")
print("\n[dev 유형별 P · R (Wilson 95% CI · D-40)]")
for i, l in enumerate(CONFIRMED):
    n = int(_dy[:, i].sum()); tp = int(((_dp[:, i] >= _th_dev[i]) & (_dy[:, i] == 1)).sum()); pp = int((_dp[:, i] >= _th_dev[i]).sum())
    rl, rh = wilson(tp, n); pl, ph = wilson(tp, pp)
    dev_cond[f"R:{l}"] = tp / n if n else float("nan"); dev_cond[f"n_R:{l}"] = n; dev_cond[f"k_R:{l}"] = tp  # v7.5 — G2 · 유형별 표
    print(f"  {l:20s} n={n:4d}  R {tp/max(n,1):.3f} [{rl:.3f}, {rh:.3f}]  P {tp/max(pp,1):.3f} [{pl:.3f}, {ph:.3f}]" + ("  측정 불가(n<30)" if n < 30 else ""))


## 7. test 채점 (후보당 한 번 · 보고용) — 가장 좋은 시드 · dev threshold

후보 선택은 섹션 6의 dev 평균으로 한다. 여기 수치는 선택에 쓰지 않는다.
- `scored`: 유형별 P/R/F1(dev threshold · 0.5) · ROC-AUC · PR-AUC
- 조건 버킷: 확정 6종 중 하나라도 threshold를 넘으면 「플래그」
- 법별 · 조건별 · 호별 적중률

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
if torch.cuda.is_available():
    best_model = best_model.cuda()
TH_BEST = seed_results[BEST_SEED]["thresholds"]
TH = TH_ROBUST if THRESHOLD_MODE == "robust" else TH_BEST
print(f"적용 threshold: {THRESHOLD_MODE} · 모델: 시드 {BEST_SEED}")

# ── v7.4: 🔴 test 는 FINAL_TEST 일 때만 채점한다 (D-175)
micro_dev = micro_05 = macro_meas = f5ho = None
tm, test_label, bucket_summary, law_f1, ho_hit, gate_rows = {}, {}, {}, {}, {}, []
if not FINAL_TEST:
    print("🔒 test 봉인 — FINAL_TEST=False. 실험 비교는 섹션 6 끝의 dev 조건 지표로 한다 (D-175)")
else:
    print("🔴 최종 측정 — 이 결과로 설정 · 데이터를 다시 고르지 않는다 (D-175)")
    th_vec = np.array([TH[l] for l in CONFIRMED])

    scored_rows = BUCKETS["scored"]
    y_te = np.stack([to_multihot(r["labels"]) for r in scored_rows])[:, :N_CONF].astype(int)
    p_te_all = predict_probs(best_model, scored_rows)
    p_te = p_te_all[:, :N_CONF]
    pred_dev = (p_te >= th_vec).astype(int)
    pred_05 = (p_te > 0.5).astype(int)
    tm = conf_metrics(np.hstack([y_te, np.zeros((len(y_te), len(LABEL_LIST) - N_CONF), int)]), p_te_all)

    print(f"[test scored {len(scored_rows)}행 · 시드 {BEST_SEED}]")
    print(f"{'유형':20s} {'정답':>5s} {'th':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'F1@0.5':>7s} {'ROC':>6s} {'PR-AUC':>7s}")
    test_label = {}
    for i, l in enumerate(CONFIRMED):
        n = int(y_te[:, i].sum())
        p_, r_, f_, _ = precision_recall_fscore_support(y_te[:, i], pred_dev[:, i], average="binary", zero_division=0)
        f05 = f1_score(y_te[:, i], pred_05[:, i], zero_division=0)
        roc, pr = tm.get(f"rocauc_{l}"), tm.get(f"prauc_{l}")
        note = "  측정 불가(n<30)" if n < 30 else ""
        test_label[l] = {"n": n, "th": TH[l], "P": p_, "R": r_, "F1": f_, "F1@0.5": f05, "rocauc": roc, "prauc": pr}
        fmt = lambda v: f"{v:6.3f}" if v is not None else "   N/A"
        tp_ = int((pred_dev[:, i] & y_te[:, i]).sum()); pp_ = int(pred_dev[:, i].sum())
        rci, pci = wilson(tp_, n), wilson(tp_, pp_)
        test_label[l].update({"R_ci": rci, "P_ci": pci})
        print(f"{l:20s} {n:5d} {TH[l]:6.3f} {p_:6.3f} {r_:6.3f} {f_:6.3f} {f05:7.3f} {fmt(roc)} {fmt(pr):>7s}{note}  R CI [{rci[0]:.2f},{rci[1]:.2f}] P CI [{pci[0]:.2f},{pci[1]:.2f}]")

    _, _, micro_dev, _ = precision_recall_fscore_support(y_te, pred_dev, average="micro", zero_division=0)
    _, _, micro_05, _ = precision_recall_fscore_support(y_te, pred_05, average="micro", zero_division=0)
    gi = [CONFIRMED.index("소비자_기만"), CONFIRMED.index("후기_체험기_기만")]
    f5ho = f1_score(y_te[:, gi].any(1), pred_dev[:, gi].any(1), zero_division=0)
    measurable = [l for l in CONFIRMED if test_label[l]["n"] >= 30]
    macro_meas = float(np.mean([test_label[l]["F1"] for l in measurable]))
    print(f"\nmicro F1 {micro_dev:.3f} (0.5 고정 {micro_05:.3f}) · 측정 가능 {len(measurable)}종 macro F1 {macro_meas:.3f} · "
          f"5호 묶음 F1 {f5ho:.3f} · macro PR-AUC {tm['macro_prauc']:.3f} · macro ROC-AUC {tm['macro_rocauc']:.3f}")

    # ── v7.2 참고: 두 threshold 방식 비교 (같은 모델 · 보고용)
    def th_summary(thd):
        tv = np.array([thd[l] for l in CONFIRMED])
        pr = (p_te >= tv).astype(int)
        _, _, mf, _ = precision_recall_fscore_support(y_te, pr, average="micro", zero_division=0)
        mac = np.mean([f1_score(y_te[:, CONFIRMED.index(l)], pr[:, CONFIRMED.index(l)], zero_division=0) for l in measurable])
        neg = float((predict_probs(best_model, BUCKETS["neg"])[:, :N_CONF] >= tv).any(1).mean())
        return mf, mac, float(pr.any(1).mean()), neg
    print("\n[threshold 방식 비교 · 같은 모델]  micro F1 / 측정가능 macro F1 / 위반 탐지 재현율 / 음성 오판정")
    for name, thd in [("best_seed", TH_BEST), ("robust", TH_ROBUST)]:
        mf, mac, vr, ng = th_summary(thd)
        print(f"  {name:10s} {mf:.3f} / {mac:.3f} / {vr*100:.1f}% / {ng*100:.1f}%" + ("   ← 적용" if thd is TH else ""))

    # ── 조건 버킷
    bucket_probs = {b: predict_probs(best_model, BUCKETS[b])[:, :N_CONF] for b in ["neg", "D", "M", "pending", "aux"]}
    flag = lambda p: float((p >= th_vec).any(axis=1).mean()) if len(p) else float("nan")
    bucket_summary = {"neg_fpr": flag(bucket_probs["neg"]), "D_flag": flag(bucket_probs["D"]),
                      "M_flag": flag(bucket_probs["M"]), "pending_recall": flag(bucket_probs["pending"]),
                      "violation_recall": float((pred_dev.any(axis=1)).mean())}
    print("\n=== 조건 버킷 (dev threshold · 확정 6종 중 하나라도) ===")
    for k, b in [("neg_fpr", "neg"), ("D_flag", "D"), ("M_flag", "M"), ("pending_recall", "pending")]:
        print(f"  {k:16s} {bucket_summary[k]*100:6.1f}%  (n={len(BUCKETS[b])})")
    print(f"  위반 탐지 재현율  {bucket_summary['violation_recall']*100:6.1f}%  (scored {len(scored_rows)})")
    print(f"  aux(확정 6종 밖 위반) 플래그율 {flag(bucket_probs['aux'])*100:.1f}%  (n={len(BUCKETS['aux'])} · 참고)")

    # ── 🆕 10-01 팀장 채점 기준 — C·A·B 양성 · L 음성 · D 별도 지표 · M 채점 안 함
    _pos_k = int(pred_dev.any(axis=1).sum()) + int((bucket_probs["pending"] >= th_vec).any(axis=1).sum())
    _pos_n = len(scored_rows) + len(BUCKETS["pending"])
    _neg_k = int((bucket_probs["neg"] >= th_vec).any(axis=1).sum()); _neg_n = len(BUCKETS["neg"])
    _d_k = int((bucket_probs["D"] >= th_vec).any(axis=1).sum()); _d_n = len(BUCKETS["D"])
    bucket_summary.update({"team_pos_recall": _pos_k / _pos_n if _pos_n else float("nan"),
                           "team_L_fpr": _neg_k / _neg_n if _neg_n else float("nan"),
                           "team_D_flag": _d_k / _d_n if _d_n else float("nan"),
                           "team_n": {"pos": _pos_n, "L": _neg_n, "D": _d_n, "M": len(BUCKETS["M"])}})
    print("\n=== 팀장 채점 기준 (C·A·B 양성 · L 음성 · D 별도 · M 채점 안 함) ===")
    for name, k, n, note in [("양성 탐지 재현율 (C·A·B)", _pos_k, _pos_n, "높을수록 좋다"),
                             ("음성 오판정률 (L)", _neg_k, _neg_n, "낮을수록 좋다"),
                             ("D 플래그율 (별도 지표)", _d_k, _d_n, "낮을수록 좋다 — 인코더는 not_claim 을 내지 않는다(D-275)")]:
        if n:
            lo_, hi_ = wilson(k, n)
            print(f"  {name:22s} {k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:.1f}~{hi_ * 100:.1f}%]  {note}{'  측정 불가(n<30)' if n < 30 else ''}")
    print(f"  M (채점 안 함) n={len(BUCKETS['M'])} · 위 M_flag 는 참고")
    _aux_k = int((bucket_probs["aux"] >= th_vec).any(axis=1).sum()); _aux_n = len(BUCKETS["aux"])
    if _pos_n + _aux_n:
        print(f"  참고 — 확정 6종 밖 위반(aux)까지 넣은 탐지율 {(_pos_k + _aux_k) / (_pos_n + _aux_n) * 100:.1f}% ({_pos_k + _aux_k}/{_pos_n + _aux_n})")

    if LABEL_UNIT == "ho" and len(BUCKETS["pending"]):   # 🆕 v8.1 — 호 판만: 유형 없는 호 로짓(0.5)까지 넣으면 pending 을 얼마나 잡나
        _pu = predict_probs(best_model, BUCKETS["pending"])
        _pk = int(((_pu[:, :N_CONF] >= th_vec).any(axis=1) | (_pu[:, LABEL2ID[UNTYPED]] >= 0.5)).sum())
        bucket_summary["pending_hit_with_untyped"] = [_pk, len(BUCKETS["pending"])]
        print(f"  참고 — 유형 없는 호 로짓까지 넣은 pending 탐지 {_pk}/{len(BUCKETS['pending'])} {_pk / len(BUCKETS['pending']) * 100:.1f}%")

    # ── 🆕 v8 (D-301) 적법 문장 오탐 — 내역과 함께만 낸다
    _dflag = (bucket_probs["D"] >= th_vec).any(axis=1)
    _gf = np.array([r["id"].startswith("gf:") for r in BUCKETS["D"]], dtype=bool)
    _nc_k, _nc_n = int(_dflag[_gf].sum()), int(_gf.sum())
    _gd_k, _gd_n = int(_dflag[~_gf].sum()), int((~_gf).sum())
    bucket_summary.update({"lawful_claim": [_neg_k, _neg_n], "lawful_noclaim": [_nc_k, _nc_n], "D_guide_flag": [_gd_k, _gd_n],
                           "aux_flag": [_aux_k, _aux_n]})
    print("\n=== 적법 문장 오탐 (D-301 · 내역과 함께) ===")
    for name, k, n in [("주장 있음 (조건 L)", _neg_k, _neg_n), ("주장 없음 (수정문구 판 D)", _nc_k, _nc_n)]:
        if n:
            lo_, hi_ = wilson(k, n)
            print(f"  {name:24s} {k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:.1f}~{hi_ * 100:.1f}%]{'  측정 불가(n<30)' if n < 30 else ''}")
        else:
            print(f"  {name:24s} 행 없음")
    if _gd_n:
        print(f"  참고 — 해설서 위반문구 D {_gd_k}/{_gd_n} {_gd_k / _gd_n * 100:.1f}%  (적법 문장이 아니다 — 기관이 삭제를 지시한 문구 · 오탐률로 세지 않는다)")

    # ── 법별 · 조건별 · 호별
    hit = (pred_dev & y_te).any(axis=1)
    law_f1 = {}
    print("\n=== 법별 micro F1 ===")
    for law in sorted({lw for r in scored_rows for lw in r["laws"]}):
        idx = [k for k, r in enumerate(scored_rows) if law in r["laws"]]
        _, _, f, _ = precision_recall_fscore_support(y_te[idx], pred_dev[idx], average="micro", zero_division=0)
        law_f1[law] = f
        print(f"  {LAW_NAMES.get(law, law):12s} n={len(idx):5d}  micro F1={f:.3f}")
    print("\n=== 조건별 적중률 (정답 유형 중 하나라도) ===")
    for c, name in [("C", "C 표현 자체"), ("A", "A 조성·지위"), ("B", "B 실증 필요"), (None, "조건 없음")]:
        idx = [k for k, r in enumerate(scored_rows) if r["condition"] == c]
        if idx:
            print(f"  {name:16s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}%")
    ho_hit = {}
    print("\n=== 호별 적중률 (n<30 참고용) ===")
    for ho in sorted({h for r in scored_rows for h in r["hos"]}):
        idx = [k for k, r in enumerate(scored_rows) if ho in r["hos"]]
        ho_hit[ho] = [float(hit[idx].mean()), len(idx)]
        lo_, hi_ = wilson(int(hit[idx].sum()), len(idx))
        print(f"  {ho:28s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}% [{lo_*100:.0f}~{hi_*100:.0f}%]{'' if len(idx) >= 30 else '  측정 불가(호 n<30 · D-40)'}")

## 8. 기준선 B(사전 단독판정) + G1 게이트 — 확정 6종 · scored

G1: 모델 Recall(dev threshold) ≥ 기준선 B Recall + 0.15. 정답 30건 미만은 참고용(D-40).
🔄 v8 — 사전이 506종이 되고 단독판정 자격이 줄었다(D-311 `비주장문맥`). 기준선 B 는 올린 사전으로 다시 잡힌다 — 사전 sha 를 로그에 남긴다.
기준선은 `원문 in text`(정규화 없음)라 판정기 B(`app/dictmatch.py`)와 수가 조금 다를 수 있다.


In [ ]:
if FINAL_TEST:
    banned_terms_rows = load_jsonl("banned_terms.jsonl")
    solo = [r for r in banned_terms_rows if r.get("단독판정") is True]
    term_to_labels = [(s, set(r.get("유형", [])) & set(CONFIRMED)) for r in solo for s in r.get("원문", [])]
    print(f"banned_terms {len(banned_terms_rows)}건 · 단독판정 {len(solo)}건 · 비주장문맥 {sum(1 for r in banned_terms_rows if r.get('신뢰도') == '비주장문맥')}건 · sha {BANNED_SHA[:12]}")

    def rule_predict(text):
        out = set()
        for s, ls in term_to_labels:
            if s and s in text:
                out |= ls
        return out

    rule_preds = [rule_predict(r["text"]) for r in scored_rows]
    GATE_MARGIN, MIN_N = 0.15, 30
    gate_rows = []
    print(f"\n{'유형':20s} {'기준선B':>8s} {'모델R':>7s} {'통과선':>7s} {'n':>5s}  판정")
    for i, l in enumerate(CONFIRMED):
        idx = [k for k in range(len(scored_rows)) if y_te[k, i]]
        n = len(idx)
        if not n:
            continue
        b = sum(l in rule_preds[k] for k in idx) / n
        m = float(pred_dev[idx, i].mean())
        verdict = f"참고용(n<{MIN_N})" if n < MIN_N else ("통과" if m >= b + GATE_MARGIN else "미달")
        gate_rows.append({"label": l, "baseline_recall": b, "model_recall": m, "pass_line": b + GATE_MARGIN, "n": n, "verdict": verdict})
        print(f"{l:20s} {b:8.3f} {m:7.3f} {b + GATE_MARGIN:7.3f} {n:5d}  {verdict}")
else:
    print("🔒 기준선 B 게이트는 최종 측정 때 함께 본다 (test 사용 · D-175)")


## 9. 실험 누적 — 후보 비교는 dev 평균 ± 표준편차로

표의 dev 열로 후보를 고른다. Drive에 `copylane_v8_experiment_log.json`으로 누적 저장한다(v7 로그와 평가셋이 달라 파일을 나눴다).
비교 기준은 `all · 낱말 포함`(머리말의 사전 고정 기준) — 그 실행이 로그에 있어야 자동 판정이 나온다.


In [ ]:
EXPERIMENT_LOG[exp_key] = {
    "golden_sha256": GOLDEN_SHA, "extra_train": EXTRA_TRAIN_FILES, "eval_only": EVAL_ONLY_FILES, "extra_sha256": EXTRA_SHA,
    "extra_rows": len(extra_rows), "extra_holdout_rows": {f: len(r) for f, r in extra_holdout.items()}, "seeds": SEEDS, "best_seed": BEST_SEED,
    "dev_macro_prauc_mean": float(np.mean(devs)), "dev_macro_prauc_std": float(np.std(devs)),
    "dev_by_seed": {str(s): seed_results[s]["dev"] for s in SEEDS}, "dev_cond": dev_cond,
    "best_epoch_by_seed": {str(s): seed_results[s]["best_epoch"] for s in SEEDS},
    "thresholds": TH, "train_rows": len(train_rows), "dev_rows": len(dev_rows), "notebook": "v8.1", "final_test": FINAL_TEST,
    "label_unit": LABEL_UNIT, "train_labels": list(TRAIN_LABELS),
    "banned_terms_sha256": BANNED_SHA, "use_word_rows": USE_WORD_ROWS, "drop_untyped_pos": DROP_UNTYPED_POS,
    "train_by_cond": TRAIN_BY_COND, "train_role_counts": dict(ROLE_COUNTS),
}
if FINAL_TEST:
    EXPERIMENT_LOG[exp_key]["test"] = {"micro_f1": micro_dev, "micro_f1_at05": micro_05, "macro_f1_measurable": macro_meas,
        "f1_5ho": f5ho, "macro_prauc": tm["macro_prauc"], "macro_rocauc": tm["macro_rocauc"],
        **bucket_summary, "law_f1": law_f1, "label": test_label, "ho_hit": ho_hit, "gate": gate_rows}

pct = lambda v: f"{v*100:5.1f}%" if isinstance(v, (int, float)) and v == v else "    —"
hold_cols = sorted({k for m in EXPERIMENT_LOG.values() for k in m.get("dev_cond", {}) if k.startswith("hold:")})
short = lambda k: k.split(":", 1)[1].replace("proto_", "")
print(f"| {'실험':58s} | {'dev PR-AUC':>13s} | {'dev 재현율':>8s} | {'dev 음성FPR':>9s} | " + "".join(f"{short(c):>16s} | " for c in hold_cols) + "비고 |")
for k, m in sorted(EXPERIMENT_LOG.items(), key=lambda kv: -kv[1]["dev_macro_prauc_mean"]):
    dc = m.get("dev_cond", {})
    note = "최종 test 채점" if m.get("final_test") else ("탐색치(D-175 이전 · test 봤음)" if "test" in m else "")
    print(f"| {k:58s} | {m['dev_macro_prauc_mean']:.3f} ± {m['dev_macro_prauc_std']:.3f} | {pct(dc.get('dev_violation_recall')):>8s} | "
          f"{pct(dc.get('dev_neg_fpr')):>9s} | " + "".join(f"{pct(dc.get(c)):>16s} | " for c in hold_cols) + f"{note} |")
print("(홀드아웃 칸: 탐지율은 높을수록 · 오탐률은 낮을수록 좋다. 같은 행으로 잰 수라 실행끼리 비교된다)")

# ── v8 후보 비교 — 같은 골든 · 같은 백본끼리만. 기준은 all · 낱말 포함(BASE_KEY)
def _wil(k, n):
    lo, hi = wilson(k, n)
    return f"{k / n * 100:5.1f}% [{lo * 100:4.1f}~{hi * 100:4.1f}]" if n else "    —"

def cand_name(k):
    # 'kcbert|이유=all|focal(…)|+proto…|v8|낱말제외|호학습' → 'all·낱말제외·호학습'
    tail = k.split("|v8", 1)[1] if "|v8" in k else ""
    return k.split("|")[1].replace("이유=", "") + tail.replace("|", "·")

CANDS = {k: m for k, m in EXPERIMENT_LOG.items()
         if k.startswith(f"{BACKBONE}|이유=") and str(m.get("notebook", "")).startswith("v8") and m.get("golden_sha256") == GOLDEN_SHA}
print(f"\n=== v8 후보 비교 ({len(CANDS)}건 · 기준 {cand_name(BASE_KEY) if BASE_KEY in CANDS else '없음 — 기준 실행(' + cand_name(BASE_KEY) + ')을 먼저 돌린다'}) ===")
print(f"{'유형별 dev 재현율 (Wilson 95%)':34s} " + " | ".join(f"{cand_name(k):>24s}" for k in CANDS))
for l in CONFIRMED:
    row = []
    for k, m in CANDS.items():
        dc = m.get("dev_cond", {})
        row.append(_wil(dc.get(f"k_R:{l}", 0), dc.get(f"n_R:{l}", 0)) if f"n_R:{l}" in dc else "    —")
    n_ = next((m["dev_cond"].get(f"n_R:{l}") for m in CANDS.values() if f"n_R:{l}" in m.get("dev_cond", {})), 0)
    print(f"{l:20s} n={n_:<4d}{'(측정 불가)' if n_ < 30 else '          '} " + " | ".join(f"{c:>24s}" for c in row))
# 🆕 v8.1 — 유형 없는 위반(dev 묶음 · 학습 안 함) 탐지율 · 참고
_ur = [_wil(m["dev_cond"]["k_untyped_hit"], m["dev_cond"]["n_untyped"]) if m.get("dev_cond", {}).get("n_untyped") else "    —" for m in CANDS.values()]
print(f"{'유형 없는 위반 탐지 (참고)':34s} " + " | ".join(f"{c:>24s}" for c in _ur))

if BASE_KEY in CANDS:
    b = CANDS[BASE_KEY]["dev_cond"]; b_au = CANDS[BASE_KEY]["dev_macro_prauc_mean"]
    hk = next((c for c in b if c.startswith("hold:") and c.endswith(":탐지율")), None)
    print(f"\n=== 사전 고정 기준 (이번 실행 ↔ 기준 {cand_name(BASE_KEY)} · G5 는 로컬 run_judge_dist 로 본다) ===")
    for k, m in CANDS.items():
        if k != exp_key or k == BASE_KEY:   # 🔄 v8.1 — 이번 실행만 그 기준과 견준다
            continue
        d = m["dev_cond"]; res = []
        g1 = (b["dev_violation_recall"] - d["dev_violation_recall"]) * 100
        res.append(("G1 재현율 하락 ≤ 2%p", g1 <= 2.0, f"하락 {g1:.1f}%p ({b.get('k_pos_hit', '?')}→{d.get('k_pos_hit', '?')}줄)"))
        for l in CONFIRMED:   # v8 — 기준 실행의 dev 양성이 30 이상인 유형만 (D-40)
            if b.get(f"n_R:{l}", 0) >= 30 and f"R:{l}" in d:
                g2 = (b[f"R:{l}"] - d[f"R:{l}"]) * 100
                res.append((f"G2 {l} 하락 ≤ 5%p", g2 <= 5.0, f"하락 {g2:.1f}%p"))
        g3 = d.get("k_neg_flag", 0) - b.get("k_neg_flag", 0)
        res.append(("G3 적법 오판정 증가 ≤ 1줄", g3 <= 1, f"증가 {g3:+d}줄"))
        if hk and hk in d:
            g4 = (b[hk] - d[hk]) * 100
            res.append(("G4 4호 탐지 하락 ≤ 5%p", g4 <= 5.0, f"하락 {g4:.1f}%p"))
        b2 = (b_au - m["dev_macro_prauc_mean"])
        res.append(("B2 PR-AUC 하락 ≤ 0.02", b2 <= 0.02, f"변화 {-b2:+.3f}"))
        ok = all(r[1] for r in res)
        print(f"\n[{cand_name(k)}] → {'탈락 기준 통과 (G5 로컬 확인)' if ok else '탈락'}")
        for name, passed, val in res:
            print(f"  {'✅' if passed else '❌'} {name:26s} {val}")
        if LABEL_UNIT == "ho":   # 🆕 v8.1 — 호 판을 고르는 조건(머리말 · 돌리기 전에 정했다): PR-AUC +0.015 이상 또는 음성 오판정 3줄 이상 감소
            _gain = m["dev_macro_prauc_mean"] - b_au; _fewer = b.get("k_neg_flag", 0) - d.get("k_neg_flag", 0)
            _pick = ok and (_gain >= 0.015 or _fewer >= 3)
            print(f"  고르는 조건 — PR-AUC {_gain:+.3f} (≥ +0.015) · 음성 오판정 {-_fewer:+d}줄 (≤ −3) → {'호 판' if _pick else '유형 판 유지'}")

print("\n=== 유형별 dev PR-AUC (시드 평균) ===")
keys = list(CANDS) or list(EXPERIMENT_LOG)
for l in CONFIRMED:
    cells_ = []
    for k in keys:
        dv = [v.get(f"prauc_{l}") for v in EXPERIMENT_LOG[k]["dev_by_seed"].values()]
        dv = [v for v in dv if v is not None]
        cells_.append(f"{np.mean(dv):.3f}" if dv else "N/A")
    print(f"{l:20s} " + "  ".join(f"{c:>7s}" for c in cells_))
print("(열 순서:", [cand_name(k) for k in keys], ")")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # v7.2 — 덮어쓰기 방지: 디스크의 기존 로그를 다시 읽어 합친 뒤 저장한다
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, encoding="utf-8") as f:
            on_disk = json.load(f)
        on_disk.update(EXPERIMENT_LOG)
        EXPERIMENT_LOG = on_disk
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(EXPERIMENT_LOG, f, ensure_ascii=False, indent=2, default=float)
    print(f"로그 {len(EXPERIMENT_LOG)}건")
    print("\nDrive 저장:", LOG_PATH)
except Exception as e:
    print("Drive 저장 건너뜀:", e)

## 10. (선택) 시드 폴더 정리

다음 후보를 돌리기 전에 디스크가 부족하면 이 셀로 이번 후보의 시드 폴더를 지운다. 저장(섹션 11)을 할 거면 먼저 저장하고 지운다.

In [ ]:
CLEAN_RUN_DIR = False
if CLEAN_RUN_DIR:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    print("삭제:", RUN_DIR)
else:
    print("유지:", RUN_DIR, "(CLEAN_RUN_DIR=True로 바꾸면 삭제)")

## 11. 모델 저장 (앱 연결용) — `SAVE_THIS_RUN = True`일 때만

- 가장 좋은 시드의 가중치에서 **분류층을 확정 6종 로짓만 남기도록 잘라** 6로짓 모델로 저장한다(보조 로짓 제거).
- 폴더 이름은 앱 로더(`app/encoder.py`)에 맞춘 `copylane-encoder-kcbert-final`. `config.json`의 `id2label`과 `label_scheme.json`의 `label_list`가 같은 순서다.
- `label_thresholds`는 **dev에서 고른 값**이다. 앱 로더는 `확률 >= threshold`로 후보를 낸다.
- 🆕 v8 — `label_scheme.json` 에 **조건별 학습 행 수 · 뺀 행 수 · 사전 sha** 를 남긴다. 그래프에 붙일 때 「새 판 · M 제외」로 학습했는지를 이 값으로 확인한다(전달 10-05 §4-3).
- 저장 zip 은 `<RUN_NAME>.zip` 하나다(Colab · Drive 같은 이름). 🔴 압축은 **저장소 밖**이나 `models/` 아래에 푼다 — `docs/psj/e2e_prototype` 안에 두지 않는다(.gitignore 10-01).


In [ ]:
def truncate_head(src_model, n_keep):
    """분류층(out_features == 전체 로짓 수인 마지막 Linear)을 앞 n_keep 로짓만 남긴 새 모델을 만든다."""
    cfg = src_model.config
    id2l = {i: CONFIRMED[i] for i in range(n_keep)}
    new = AutoModelForSequenceClassification.from_config(
        cfg.__class__.from_dict({**cfg.to_dict(), "num_labels": n_keep, "id2label": id2l,
                                 "label2id": {v: k for k, v in id2l.items()}}))
    sd = src_model.state_dict()
    head = [n for n, m in src_model.named_modules()
            if isinstance(m, torch.nn.Linear) and m.out_features == src_model.config.num_labels][-1]
    new_sd = {}
    for k, v in sd.items():
        if k == f"{head}.weight":
            new_sd[k] = v[:n_keep].clone()
        elif k == f"{head}.bias":
            new_sd[k] = v[:n_keep].clone()
        else:
            new_sd[k] = v
    missing, unexpected = new.load_state_dict(new_sd, strict=False)
    assert not [m for m in missing if "position_ids" not in m], f"빠진 가중치: {missing}"
    return new

if not SAVE_THIS_RUN:
    print("SAVE_THIS_RUN=False → 저장 건너뜀")
elif LABEL_UNIT == "ho":
    print("호 판 — 앱 로더 형식(유형 6로짓)으로 저장하지 않는다. 접기(호 → 유형)를 추론 때 해야 해서다.")
    print("이 판을 고르게 되면 저장 형식(호 로짓 + 접기 표를 함께 · 앱 쪽에서 접기)을 팀장과 정한 뒤 다시 돌려 저장한다.")
else:
    SAVE_DIR = "./copylane-encoder-kcbert-final"
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    src = best_model.cpu().float()
    final = truncate_head(src, N_CONF) if len(LABEL_LIST) > N_CONF else src
    chk = dev_rows[:64]
    diff = np.abs(predict_probs(src, chk)[:, :N_CONF] - predict_probs(final, chk)).max()
    assert diff < 1e-5, f"잘라낸 모델 출력이 다르다: {diff}"
    final.save_pretrained(SAVE_DIR, safe_serialization=True)
    tokenizer.save_pretrained(SAVE_DIR)
    assert os.path.exists(f"{SAVE_DIR}/tokenizer.json"), "tokenizer.json이 없다 (앱 로더 필수 파일)"
    with open(f"{SAVE_DIR}/label_scheme.json", "w", encoding="utf-8") as f:
        json.dump({
            "label_list": CONFIRMED,
            "label_thresholds": TH,
            "backbone": MODEL_NAME, "notebook": "v8.1 (3차 골든 · D-296 개정 2)", "label_unit": LABEL_UNIT,
            "experiment": exp_key, "seed": BEST_SEED,
            "best_epoch": seed_results[BEST_SEED]["best_epoch"],
            "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS), "banned_terms_sha256": BANNED_SHA,
            "train_rows": len(train_rows), "train_by_cond": TRAIN_BY_COND, "train_role_counts": dict(ROLE_COUNTS),
            "train_rule": "조건 M 제외 · D · L 음성 · C · A · B · 조건 없음은 유형이 양성 (D-296 개정 2)"
                          + (" · 유형 없는 위반 제외" if DROP_UNTYPED_POS else " · 유형 없는 위반은 음성") + ("" if USE_WORD_ROWS else " · 낱말 행 제외"),
            "use_word_rows": USE_WORD_ROWS, "drop_untyped_pos": DROP_UNTYPED_POS, "aux_labels": AUX if USE_AUX_LOGITS else [],
            "threshold_mode": THRESHOLD_MODE,
            "thresholds_by_seed": {str(s_): seed_results[s_]["thresholds"] for s_ in SEEDS},
            "threshold_note": ("시드 3개 dev 예측의 평균 F1 최대" if THRESHOLD_MODE == "robust" else "가장 좋은 시드의 dev F1 최대") + " · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
            "scope_note": "확정 6종(D-232) · 유형 단위 출력(D-131). 판정 대상 아님(D-275) · low_conf(M) · 화장품법 13①4 등 유형 없는 위반은 이 인코더가 내지 않는다.",
            "test_report": ({"micro_f1": micro_dev, **bucket_summary} if FINAL_TEST else None),
            "final_test": FINAL_TEST, "dev_cond": dev_cond,
            "extra_train": EXTRA_TRAIN_FILES, "extra_sha256": {f: EXTRA_SHA[f] for f in EXTRA_TRAIN_FILES}, "extra_rows": len(extra_rows),
        }, f, ensure_ascii=False, indent=2)
    print("저장:", sorted(os.listdir(SAVE_DIR)), f"(6로짓 확인 · 최대 확률 차 {diff:.1e})")
    zip_path = shutil.make_archive(RUN_NAME, "zip", SAVE_DIR)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dst = f"/content/drive/MyDrive/{RUN_NAME}.zip"
    shutil.copy(zip_path, dst)
    print("드라이브 저장:", dst)